# MAG Access Guide

This tutorial walks you through obtaining access, configuring your
environment, discovering the models your credential can actually use, and
making a minimal, reproducible Model Access Gateway (MAG) request.

> **Before you begin:** MAG access and model availability are
> project-specific. Do not send sensitive or unapproved project data. Keep
> personal access tokens (PATs) out of notebooks, source control,
> screenshots, and logs.

**Prerequisites:** Python 3.11+ with the packages in `requirements.txt`
installed (`jupyterlab`, `requests`, `openai`).

Sources: [MAG documentation](https://docs.amsc.energy.gov/model-access-gateway)
Prepared: 2026-08-24

## What you will do

1. Confirm the access path and obtain a project-scoped MAG PAT.
2. Put the PAT in an environment variable, not this notebook.
3. List the live models available to your credential.
4. Select a chat model against task, stability, latency, and capability
   requirements.
5. Make one small chat-completions request and inspect the returned model ID.
6. Optionally repeat the call through the OpenAI Python SDK.

The API examples use MAG's OpenAI-compatible `/v1` interface. Run the cells
in order — each one builds on names defined earlier.

## Obtain access

To use Model Access Gateway (MAG), you must have an active project with MAG
access approved.

General onboarding steps:

- Obtain an American Science Cloud (AmSC) account.
- Request an AmSC project or join an existing approved project.
- Sign in to the MAG portal via the
  [MAG documentation](https://docs.amsc.energy.gov/model-access-gateway),
  switch to your project, and generate a Personal Access Token (PAT).
- Copy the PAT immediately when displayed (it is only shown once). If lost,
  generate a replacement.

## Set up credentials safely

In a terminal **before** launching Jupyter, set the PAT for that terminal
session. Do not paste a real token into the cell below or save it in the
notebook.

```bash
export AMSC_I2_API_KEY='paste-your-PAT-here'
jupyter lab
```

Jupyter only inherits the variable from the terminal that launched it. If the
next cell reports a missing key, set the variable and restart Jupyter from
that same terminal.

For a longer-lived setup, use your institution-approved secret manager or
shell-profile approach. Treat the PAT like a password; revoke and replace it
if exposed.

In [ ]:
import os

API_BASE = "https://i2-api.genesis.american-science-cloud.org/v1"
API_KEY = os.environ.get("AMSC_I2_API_KEY")

if not API_KEY:
    raise RuntimeError(
        "AMSC_I2_API_KEY is not set. Set it in a terminal, then restart "
        "Jupyter from that same terminal."
    )

headers = {"Authorization": f"Bearer {API_KEY}"}
print("MAG PAT found in the environment.")

## Discover the models assigned to your project

Do not copy a model ID from a reference card and assume it is available. The
`/v1/models` response is the authoritative, current list for your credential,
and it may change over time.

In [ ]:
import requests

response = requests.get(f"{API_BASE}/models", headers=headers, timeout=30)

if response.status_code in (401, 403):
    raise RuntimeError(
        f"MAG rejected the credential (HTTP {response.status_code}). Check "
        "that the PAT is current, unrevoked, and scoped to a project with "
        "MAG access approved."
    )
response.raise_for_status()

model_records = response.json().get("data", [])
model_ids = sorted(
    record.get("id") or record.get("model_name")
    for record in model_records
    if record.get("id") or record.get("model_name")
)

if not model_ids:
    raise RuntimeError(
        "The credential is valid but no models are assigned to it. Confirm "
        "the active project and its approved model list in the MAG portal."
    )

print(f"{len(model_ids)} models available to this credential:")
print("\n".join(model_ids))

## Choose a model

Start with the lowest level of autonomy that solves the problem. For asking
questions, extracting structure, summarizing approved material, or testing
prompts, a direct MAG call is enough. Use a conventional script or workflow
for deterministic transformations; introduce an agent only when
interpretation, adaptive planning, or tool selection is genuinely needed.

Choose only from the live list printed above, then use this decision guide:

| Need | Prefer | What to verify |
|---|---|---|
| Fast, high-volume classification or simple extraction | A small/fast chat model | Quality on representative approved examples, latency, output format |
| General analysis, summarization, or structured drafting | A balanced chat model | Factual checks, instruction following, cost/allocation behavior |
| Difficult coding, multi-step reasoning, or high-stakes review support | A stronger model; keep a human reviewer | Evaluation set, failure modes, tool/data permissions |
| Retrieval or semantic search | An embedding model via `/v1/embeddings` | Embedding dimension, retrieval quality, language/domain fit |
| A reproducible workflow | An exact versioned model ID | Version pinning, regression tests, response `model` field |
| The newest supported behavior | A family alias | That the alias can change underlying versions |

Aliases such as `claude-sonnet` can follow the newest version in a family.
Pin an exact version (for example, a versioned ID returned by `/v1/models`)
when repeatability matters. Always record the `model` property from a
response, and evaluate candidates on a small approved test set before moving
to a broader workflow.

The cell below applies that guidance mechanically: it drops embedding-only
IDs, then picks a preferred alias if the credential has it. Override
`PREFERRED` — or set `MODEL` directly — to follow the table above.

In [ ]:
# Embedding models cannot serve /v1/chat/completions, so exclude them.
EMBEDDING_HINTS = ("embed", "rerank")
PREFERRED = "claude-sonnet"

chat_model_ids = [
    model_id
    for model_id in model_ids
    if not any(hint in model_id.lower() for hint in EMBEDDING_HINTS)
]

if not chat_model_ids:
    raise RuntimeError(
        "No chat-capable model was found in the live list. Set MODEL "
        "manually to an ID printed in the previous cell."
    )

MODEL = PREFERRED if PREFERRED in chat_model_ids else chat_model_ids[0]

print("Chat-capable candidates:", ", ".join(chat_model_ids))
print("Selected model:", MODEL)

## Make the first API call

This cell sends one request using the `MODEL` selected above and consumes
part of the project's token allocation. The prompt is intentionally short and
non-sensitive.

Compare the requested ID with the `model` field MAG returns: when you request
an alias, the response reports the concrete version that served it. That
returned value is what you pin for a reproducible workflow.

In [ ]:
payload = {
    "model": MODEL,
    "messages": [
        {
            "role": "user",
            "content": "In one sentence, explain what an AI model is.",
        }
    ],
}

response = requests.post(
    f"{API_BASE}/chat/completions",
    headers={**headers, "Content-Type": "application/json"},
    json=payload,
    timeout=60,
)
response.raise_for_status()
result = response.json()

print("Requested model:", MODEL)
print("Model reported by MAG:", result.get("model"))
print()
print("Reply:")
print(result["choices"][0]["message"]["content"])

## Optional: use the OpenAI Python SDK

MAG provides an OpenAI-compatible interface. Applications using the `openai`
SDK, or frameworks that accept a custom base URL, can connect by pointing
`base_url` at MAG:

- `base_url`: `https://i2-api.genesis.american-science-cloud.org/v1`
- `api_key`: `$AMSC_I2_API_KEY`

This repeats the previous request through a different client, so it
consumes another small piece of the token allocation.

In [ ]:
from openai import OpenAI

client = OpenAI(api_key=API_KEY, base_url=API_BASE)

completion = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": "In five words, describe scientific discovery.",
        }
    ],
)

print("Reported model:", completion.model)
print("Reply:", completion.choices[0].message.content)

## Troubleshooting and next steps

- **`AMSC_I2_API_KEY is not set`:** the variable was exported after Jupyter
  started, or in a different terminal. Export it, then relaunch Jupyter from
  that terminal.
- **401/403, or an empty model list:** recheck the active project, the PAT
  value, the endpoint, and project membership. Model access is
  project-specific.
- **Model not found:** re-run the model discovery cell and choose an ID
  returned for this credential; assignments change over time.
- **Unexpected behavior:** preserve non-sensitive logs, reduce permissions
  and prompt scope, then retry with a smaller test. Do not treat fluent
  output as scientific evidence.
- **Possible token exposure:** revoke the PAT, create a replacement, and
  follow your institution's incident process.

Before expanding from this test, document the chosen model, exact
version/alias behavior, permitted data classes, expected outputs, human
review, and a small evaluation set.

## References

- American Science Cloud,
  [Model Access Gateway (MAG)](https://docs.amsc.energy.gov/model-access-gateway)

This notebook summarizes the published guidance as of the preparation date.
Follow the live documentation and your project onboarding instructions if
they differ.